# Clase 6 - Regresión Logística

Pasamos de predecir números a predecir categorías. Arrancamos viendo la
sigmoide, después un ejemplo simple con una sola variable, y cerramos con
un dataset real de clasificación binaria.

## 1. La función sigmoide

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)

plt.plot(z, sigmoide(z))
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("z")
plt.ylabel("sigmoide(z)")
plt.title("Función sigmoide")
plt.show()

Cualquier valor de `z` (positivo, negativo, grande o chico) queda aplastado
entre 0 y 1. En `z = 0` la sigmoide vale exactamente 0.5.

## 2. Clasificación binaria con una sola variable

Simulamos si algun estudiante aprueba un examen según las horas que
estudió. A más horas, mayor probabilidad de aprobar (pero no es
determinístico).

In [ ]:
rng = np.random.default_rng(seed=0)

horas_estudio = rng.uniform(0, 10, 150)
prob_real = sigmoide(horas_estudio - 5)   # probabilidad "verdadera" de aprobar
aprobo = rng.binomial(1, prob_real)

examenes = pd.DataFrame({"horas_estudio": horas_estudio, "aprobo": aprobo})

plt.scatter(examenes["horas_estudio"], examenes["aprobo"], alpha=0.6)
plt.xlabel("Horas de estudio")
plt.ylabel("Aprobó (0/1)")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

X = examenes[["horas_estudio"]]
y = examenes["aprobo"]

modelo = LogisticRegression()
modelo.fit(X, y)

print("w:", modelo.coef_[0][0])
print("b:", modelo.intercept_[0])

In [ ]:
horas_ordenadas = pd.DataFrame({"horas_estudio": np.linspace(0, 10, 200)})
prob_predicha = modelo.predict_proba(horas_ordenadas)[:, 1]   # prob. de la clase 1

plt.scatter(examenes["horas_estudio"], examenes["aprobo"], alpha=0.4, label="datos")
plt.plot(horas_ordenadas, prob_predicha, color="red", label="P(aprobar) estimada")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.xlabel("Horas de estudio")
plt.ylabel("Probabilidad de aprobar")
plt.legend()
plt.show()

In [ ]:
# ¿Qué probabilidad le da el modelo a alguien que estudió 3 horas? ¿Y 7 horas?
for horas in [3, 7]:
    p = modelo.predict_proba(pd.DataFrame({"horas_estudio": [horas]}))[0, 1]
    print(f"{horas} horas -> P(aprobar) = {p:.2f}")

## 3. Clasificación con varias variables: dataset real

Usamos el dataset de diagnóstico de cáncer de mama que trae scikit-learn:
30 variables numéricas calculadas a partir de una imagen, y la etiqueta es
si el tumor es maligno o benigno.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print(X.shape)
print(data.target_names)   # ['malignant', 'benign']
X.head()

Las 30 columnas están en escalas muy distintas (áreas vs. radios vs.
simetrías), así que esta vez sí escalamos antes de entrenar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

scaler = StandardScaler()
X_train_escalado = scaler.fit_transform(X_train)
X_test_escalado = scaler.transform(X_test)

Fíjense que ajustamos (`fit`) el scaler solo con `X_train`, y a `X_test` le
aplicamos únicamente `transform`. Si usáramos también `X_test` para
calcular la media y el desvío, estaríamos filtrando información del test
set al entrenamiento.

In [ ]:
modelo = LogisticRegression(max_iter=5000)
modelo.fit(X_train_escalado, y_train)

print("Accuracy en test:", round(modelo.score(X_test_escalado, y_test), 3))

In [ ]:
probabilidades = modelo.predict_proba(X_test_escalado)[:5]
predicciones = modelo.predict(X_test_escalado)[:5]

pd.DataFrame({
    "P(maligno)": probabilidades[:, 0].round(3),
    "P(benigno)": probabilidades[:, 1].round(3),
    "predicción": [data.target_names[p] for p in predicciones],
    "real": [data.target_names[r] for r in y_test.values[:5]],
})

## Ejercicios

**Ejercicio 1.** Cambiar la simulación de `examenes` para que la
probabilidad "verdadera" sea `sigmoide((horas_estudio - 5) * 2)` (una
sigmoide más "empinada"). Volver a entrenar y graficar. ¿La curva ajustada
queda más parecida o menos parecida a un escalón?

**Ejercicio 2.** En el ejemplo de `examenes`, probar con un umbral de
decisión distinto a 0.5: contar cuántos casos cambiarían de predicción si
usáramos 0.3 en vez de 0.5 (`prob_predicha >= 0.3`).

**Ejercicio 3.** Entrenar el modelo de cáncer de mama **sin** escalar los
datos (`X_train` y `X_test` directamente, sin `StandardScaler`) y comparar
el accuracy contra la versión escalada. ¿Cambia mucho?

**Ejercicio 4.** Buscar en `predicciones` (sobre todo el test set, no solo
las primeras 5 filas) algún caso donde el modelo se haya equivocado, y
mirar con qué probabilidad lo predijo. ¿Se equivocó con mucha o poca
confianza?

In [ ]:
# Ejercicio 1: 

In [ ]:
# Ejercicio 2: 

In [ ]:
# Ejercicio 3: 

In [ ]:
# Ejercicio 4: 